In [ ]:
import CDutils
import scipy.io as sio
import numpy as np
from scipy.io import savemat
import os
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

In [ ]:
path = r"D:\Code\Contraction_detection_manuscript_second_round\Contaction_detection\Data_sorted"
dir_list = os.listdir(path)

In [ ]:
dir_list[1]

In [ ]:
f1_list = []
cp_score_list = []
cpn_list = []
for pt in range(len(dir_list)):
    data = sio.loadmat(path + '/' + dir_list[pt])
    print(dir_list[pt])
    modulation_ds = data['tmp_y']
    label_ds = data['newClusterSig']
    # connectivity = data['Uterus']['tri'][0][0]
    # conn = connectivity.astype(int)-1

    # modulation_ds = modulation[:,::40]
    # label_ds = label[:,::40]
    ch_num = modulation_ds.shape[0]
    # conn_num = conn.shape[0]
    signal_length = modulation_ds.shape[1]
    
    modulation_ar_removal = CDutils.ar_remove_patient(modulation_ds)
    c_detection = np.zeros_like(modulation_ar_removal)
    window_length = 30
    for ch in range(ch_num):
        sig = modulation_ar_removal[ch,:]
        envelope = CDutils.rms_envelope(sig, window_length)
        c_detection[ch,:] = CDutils.rms_cd(envelope,window_length,0.4)
        
    f1_array = np.zeros((ch_num,))
    cp_score_array = np.zeros(ch_num,)
    cpn_array = np.zeros(ch_num,)
    for ch in range(ch_num):
        pre_cpds = CDutils.find_cpd(c_detection[ch,:], 1)
        gt_cpds = CDutils.find_cpd(label_ds[ch,:], 1)
        f1_array[ch] = f1_score(label_ds[ch,:], c_detection[ch,:],zero_division=1.0)
        cp_score_array[ch] = CDutils.cp_score(gt_cpds,pre_cpds,signal_length)
        # cp_score_array1[ch] = CDutils.cp_score1(gt_cpds,pre_cpds,signal_length)
        cpn_array[ch] = (np.abs(len(gt_cpds)-len(pre_cpds)))/len(gt_cpds)
    
    print(np.median(f1_array))
    print(np.median(cp_score_array))
    print(np.median(cpn_array))
    f1_list.append(np.median(f1_array))
    cp_score_list.append(np.median(cp_score_array))
    cpn_list.append(np.median(cpn_array))

   

In [ ]:
mdic = {"f1_list": f1_list,"cp_score_list": cp_score_list,"cpn_list": cpn_list}
savemat(r"4_RMS_result.mat", mdic)